## 0. Setup

In [ ]:
import os
import time
import json
import requests
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 200)

ETHERSCAN_API_KEY = os.getenv('ETHERSCAN_API_KEY')
if not ETHERSCAN_API_KEY:
    raise RuntimeError('Missing ETHERSCAN_API_KEY. Set it with: export ETHERSCAN_API_KEY="YOUR_KEY"')

BASE_URL = 'https://api.etherscan.io/api'

# Output folder
DATA_DIR = Path('../data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Setup complete. Data directory:', DATA_DIR.resolve())

## 1. Choose a wallet to test

In [ ]:
# Replace
WALLET = '0x0000000000000000000000000000000000000000'
WALLET

## 2. Fetch normal transactions (txlist)

In [ ]:
def etherscan_get(params: dict, min_delay_s: float = 0.25, timeout_s: int = 30) -> dict:
    time.sleep(min_delay_s)
    r = requests.get(BASE_URL, params=params, timeout=timeout_s)
    r.raise_for_status()
    data = r.json()
    # Etherscan error format
    if str(data.get('status')) == '0' and data.get('message') not in ('No transactions found', 'No records found'):
        raise RuntimeError(f"Etherscan error: {data.get('message')} | result={data.get('result')}")
    return data


def fetch_normal_transactions(wallet: str, startblock: int = 0, endblock: int = 99999999, sort: str = 'asc'):
    params = {
        'module': 'account',
        'action': 'txlist',
        'address': wallet,
        'startblock': startblock,
        'endblock': endblock,
        'sort': sort,
        'apikey': ETHERSCAN_API_KEY,
    }
    data = etherscan_get(params)
    result = data.get('result', [])
    return result if isinstance(result, list) else []


raw_normal = fetch_normal_transactions(WALLET)
len(raw_normal), (raw_normal[0].keys() if raw_normal else 'No txs')

## 3. Normalize normal transactions into a tidy table

In [ ]:
def to_int(x):
    try:
        return int(x)
    except Exception:
        return None


def normalize_normal_transactions(wallet: str, txs: list) -> pd.DataFrame:
    if not txs:
        return pd.DataFrame(columns=[
            'wallet','tx_hash','timestamp','block_number','from_address','to_address',
            'value_wei','value_eth','gas_used','gas_price_wei','is_error'
        ])

    df = pd.DataFrame(txs).copy()
    df['wallet'] = wallet
    df = df.rename(columns={
        'hash': 'tx_hash',
        'timeStamp': 'timestamp',
        'blockNumber': 'block_number',
        'from': 'from_address',
        'to': 'to_address',
        'value': 'value_wei',
        'gasUsed': 'gas_used',
        'gasPrice': 'gas_price_wei',
        'isError': 'is_error',
    })

    df['timestamp'] = pd.to_datetime(df['timestamp'].apply(to_int), unit='s', utc=True, errors='coerce')
    df['block_number'] = df['block_number'].apply(to_int)
    df['value_wei'] = df['value_wei'].apply(to_int)
    df['value_eth'] = df['value_wei'] / 1e18
    df['gas_used'] = df['gas_used'].apply(to_int)
    df['gas_price_wei'] = df['gas_price_wei'].apply(to_int)
    df['is_error'] = df['is_error'].astype(str)

    keep = [
        'wallet','tx_hash','timestamp','block_number','from_address','to_address',
        'value_wei','value_eth','gas_used','gas_price_wei','is_error'
    ]
    return df[keep].sort_values('timestamp')


tx_df = normalize_normal_transactions(WALLET, raw_normal)
tx_df.head()

## 5. Wallet summary table


In [ ]:
def build_wallet_summary(wallet: str, tx_df: pd.DataFrame, token_df: pd.DataFrame | None = None) -> dict:
    summary = {'wallet': wallet}

    if tx_df is None or tx_df.empty or tx_df['timestamp'].isna().all():
        summary.update({'first_tx_time': None,'last_tx_time': None,'tx_count': 0,'wallet_age_days': None})
    else:
        first_ts = tx_df['timestamp'].min()
        last_ts = tx_df['timestamp'].max()
        summary.update({
            'first_tx_time': first_ts.isoformat() if pd.notna(first_ts) else None,
            'last_tx_time': last_ts.isoformat() if pd.notna(last_ts) else None,
            'tx_count': int(tx_df.shape[0]),
            'wallet_age_days': float(((pd.Timestamp.utcnow().tz_localize('UTC') - first_ts).total_seconds())/86400.0) if pd.notna(first_ts) else None,
        })

    if token_df is not None and not token_df.empty:
        summary.update({
            'erc20_transfer_count': int(token_df.shape[0]),
            'unique_token_symbols': int(token_df['token_symbol'].nunique(dropna=True)),
        })
    else:
        summary.update({'erc20_transfer_count': 0,'unique_token_symbols': 0})

    return summary


summary = build_wallet_summary(WALLET, tx_df, token_df)
pd.DataFrame([summary])

## 6. Save outputs (raw JSON + normalized CSVs + sample summary)

In [ ]:
# Save raw JSON
raw_json_path = DATA_DIR / f"{WALLET}_raw.json"
with raw_json_path.open('w', encoding='utf-8') as f:
    json.dump({'wallet': WALLET, 'normal_transactions': raw_normal, 'token_transfers': raw_tokens}, f, indent=2)

# Save normalized tables
tx_csv_path = DATA_DIR / f"{WALLET}_transactions.csv"
tx_df.to_csv(tx_csv_path, index=False)

token_csv_path = DATA_DIR / f"{WALLET}_token_transfers.csv"
token_df.to_csv(token_csv_path, index=False)

# Update sample summary dataset
summary_csv_path = DATA_DIR / "sample_wallets.csv"
summary_df = pd.DataFrame([summary])
if summary_csv_path.exists():
    prev = pd.read_csv(summary_csv_path)
    prev = prev[prev['wallet'] != WALLET]
    out_df = pd.concat([prev, summary_df], ignore_index=True)
else:
    out_df = summary_df
out_df.to_csv(summary_csv_path, index=False)

print('✅ Saved:')
print(' -', raw_json_path.resolve())
print(' -', tx_csv_path.resolve())
print(' -', token_csv_path.resolve())
print(' -', summary_csv_path.resolve())